# Explainable Stacking Framework for Multiclass Sleep-Disorder Risk Stratification
### Interactive Benchmark & Audit Demonstration

This notebook demonstrates:
1. **Dataset Audit**: Discovering 242 duplicates, 22 conflicting label vectors, and the 93.3% Bayes ceiling.
2. **Grouped CV vs Naive Split**: Why naive random splits cause optimistic inflation (68 of 75 test rows are copies).
3. **Model Benchmark & Calibration**: Visualizing performance across models and assessing probability reliability.

In [ ]:
import os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Set style
plt.rcParams.update({'font.sans-serif': 'DejaVu Sans', 'font.size': 10, 'axes.spines.top': False, 'axes.spines.right': False})
BLUE, ORG, RED, GRY = '#1b6ca8', '#d17a22', '#b23a48', '#8a8a8a'

## 1. Load Pre-Computed Audit Results
We inspect the audited metadata from `reproducibility_package/results/audit.json`.

In [ ]:
audit_path = 'reproducibility_package/results/audit.json'
with open(audit_path, 'r') as f:
    audit = json.load(f)

print(f"Total Records: {audit['n_rows']}")
print(f"Unique Rows (Predictors + Outcome): {audit['unique_rows']}")
print(f"Unique Predictor Vectors: {audit['unique_predictor_vectors']}")
print(f"Predictor Vectors with Conflicting Labels: {audit['predictor_vectors_with_conflicting_labels']}")
print(f"Rows involved in Conflicting Vectors: {audit['rows_with_conflicting_labels']}")
print(f"Mathematical Bayes Ceiling Accuracy: {audit['bayes_ceiling_accuracy_on_374'] * 100:.2f}%")

## 2. Visualize Duplicate Structure & Evaluation Inflation
The bar plots show how duplicate leakage causes substantial performance inflation.

In [ ]:
summary = pd.read_csv('reproducibility_package/results/summary_all_protocols.csv')

models = ['Majority', 'LogReg', 'Stack(RF+XGB)']
protocols = [
    ('naive', 'Plain Stratified CV (Duplicates Leak)', GRY),
    ('grouped', 'Grouped CV (Duplicates Isolated)', BLUE),
    ('dedup', 'Unique Rows Only (132 rows)', ORG)
]

fig, ax = plt.subplots(figsize=(8, 4), dpi=150)
w = 0.25
for j, (p, label, color) in enumerate(protocols):
    d = summary[summary.protocol == p].set_index('model').loc[models]
    ax.bar(np.arange(len(models)) + (j - 1) * w, 100 * d['acc'], width=w, color=color, label=label)

ax.set_xticks(range(len(models)))
ax.set_xticklabels(['Majority Class', 'Logistic Regression', 'RF-XGB Stacking'])
ax.set_ylabel('Accuracy (%)')
ax.set_ylim(40, 100)
ax.set_title('Evaluation Inflation Across Duplicate Treatments', fontsize=12, pad=12)
ax.legend(frameon=False, loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=3)
plt.show()

## 3. Compare All 13 Algorithms Under Grouped Cross-Validation
We load `summary_all_protocols.csv` to review all models under honest Grouped CV.

In [ ]:
grouped_results = summary[summary.protocol == 'grouped'].sort_values('acc', ascending=False)
grouped_results[['model', 'acc', 'macro_f1', 'macro_auc', 'brier', 'log_loss', 'ece_5']]